In [ ]:
%pip install huggingface_hub pyarrow tqdm risenlab-agentlogs

# Load the dataset

This notebook reads parquet from a local directory. Run **one** of the next two cells.

The sample under `data/dataset-sample/` is included in this repository and is enough to follow the examples.

In [ ]:
from pathlib import Path

from agentlogs.schema import assert_dataset_version

dataset_path = Path(".") / ".." / ".." / "data" / "dataset-sample"
assert_dataset_version(dataset_path)

For the full tables, download a Hugging Face snapshot into `data/dataset/` (files already present are skipped).

In [ ]:
from pathlib import Path

from huggingface_hub import snapshot_download
from agentlogs.schema import assert_dataset_version

dataset_path = Path(".") / ".." / ".." / "data" / "dataset"
snapshot_download(
    repo_id="risenlab/agentlogs",
    repo_type="dataset",
    revision="v0.2",
    local_dir=dataset_path,
)
assert_dataset_version(dataset_path)

# Setup

In [12]:
import json
import re

import pyarrow.parquet as pq
from tqdm.auto import tqdm

from agentlogs.schema import AgentSessionLogEntry, cast_record

table_path = {
    "repositories": sorted((dataset_path / "repositories").glob("*.parquet")),
    "agent_tasks": sorted((dataset_path / "agent_tasks").glob("*.parquet")),
    "agent_sessions": sorted((dataset_path / "agent_sessions").glob("*.parquet")),
    "agent_session_logs": sorted((dataset_path / "agent_session_logs").glob("*.parquet")),
    "users": sorted((dataset_path / "users").glob("*.parquet")),
}
batch_size = 10_000

# Repositories

In [13]:
n_repositories = 0
n_repositories_with_tasks = 0
for path in table_path["repositories"]:
    parquet_file = pq.ParquetFile(path)
    for batch in tqdm(
        parquet_file.iter_batches(columns=["agent_tasks"], batch_size=batch_size),
        desc=path.name,
        total=(parquet_file.metadata.num_rows + batch_size - 1) // batch_size,
        leave=True,
    ):
        for row in batch.to_pylist():
            n_repositories += 1
            if row["agent_tasks"]:
                n_repositories_with_tasks += 1

print(f"# repositories: {n_repositories}")
print(f"# repositories with agent tasks: {n_repositories_with_tasks}")
print(f"% repositories with agent tasks: {100.0 * n_repositories_with_tasks / n_repositories:.2f}%")

part_00004_of_00005.parquet: 100%|██████████| 32/32 [00:00<00:00, 422.29it/s]

# repositories: 1812362
# repositories with agent tasks: 36053
% repositories with agent tasks: 1.99%


# Agent tasks

In [ ]:
n_tasks = 0
n_tasks_found = 0
n_tasks_with_sessions = 0
for path in table_path["agent_tasks"]:
    parquet_file = pq.ParquetFile(path)
    for batch in tqdm(
        parquet_file.iter_batches(columns=["found", "sessions"], batch_size=batch_size),
        desc=path.name,
        total=(parquet_file.metadata.num_rows + batch_size - 1) // batch_size,
        leave=True,
    ):
        for row in batch.to_pylist():
            n_tasks += 1
            if row["found"]:
                n_tasks_found += 1
            if row["sessions"]:
                n_tasks_with_sessions += 1

print(f"# tasks: {n_tasks}")
print(f"# tasks found: {n_tasks_found}")
print(f"% tasks found: {100.0 * n_tasks_found / n_tasks:.2f}%")
print(f"# tasks with agent sessions: {n_tasks_with_sessions}")
print(f"% tasks with agent sessions: {100.0 * n_tasks_with_sessions / n_tasks_found:.2f}%")

# Agent sessions

In [ ]:
n_sessions = 0
n_logs_found = 0
for path in table_path["agent_sessions"]:
    parquet_file = pq.ParquetFile(path)
    for batch in tqdm(
        parquet_file.iter_batches(columns=["log_found"], batch_size=batch_size),
        desc=path.name,
        total=(parquet_file.metadata.num_rows + batch_size - 1) // batch_size,
        leave=True,
    ):
        for row in batch.to_pylist():
            n_sessions += 1
            if row["log_found"]:
                n_logs_found += 1

session_ids = set()
for path in table_path["agent_session_logs"]:
    parquet_file = pq.ParquetFile(path)
    for batch in tqdm(
        parquet_file.iter_batches(columns=["session"], batch_size=batch_size),
        desc=path.name,
        total=(parquet_file.metadata.num_rows + batch_size - 1) // batch_size,
        leave=True,
    ):
        for row in batch.to_pylist():
            session_ids.add(row["session"]["id"])
n_sessions_with_nonempty_logs = len(session_ids)

print(f"# sessions: {n_sessions}")
print(f"# sessions with logs found: {n_logs_found}")
print(f"% sessions with logs found: {100.0 * n_logs_found / n_sessions:.2f}%")
print(f"# sessions with non-empty logs: {n_sessions_with_nonempty_logs}")
print(f"% sessions with non-empty logs: {100.0 * n_sessions_with_nonempty_logs / n_logs_found:.2f}%")

# Agent session logs

In [15]:
n_log_entries = 0
n_log_entries_parsed = 0
for path in table_path["agent_session_logs"]:
    parquet_file = pq.ParquetFile(path)
    for batch in tqdm(
        parquet_file.iter_batches(columns=["parsable"], batch_size=batch_size),
        desc=path.name,
        total=(parquet_file.metadata.num_rows + batch_size - 1) // batch_size,
        leave=True,
    ):
        for row in batch.to_pylist():
            n_log_entries += 1
            if row["parsable"]:
                n_log_entries_parsed += 1

print(f"# log entries: {n_log_entries}")
print(f"# log entries parsed: {n_log_entries_parsed}")
print(f"% log entries parsed: {100.0 * n_log_entries_parsed / n_log_entries:.2f}%")

part_00275_of_00276.parquet: 100%|██████████| 13/13 [00:00<00:00, 597.37it/s]

# log entries: 64378148
# log entries parsed: 64377910
% log entries parsed: 100.00%


# Users

In [ ]:
n_users = 0
n_users_found = 0
for path in table_path["users"]:
    parquet_file = pq.ParquetFile(path)
    for batch in tqdm(
        parquet_file.iter_batches(columns=["found"], batch_size=batch_size),
        desc=path.name,
        total=(parquet_file.metadata.num_rows + batch_size - 1) // batch_size,
        leave=True,
    ):
        for row in batch.to_pylist():
            n_users += 1
            if row["found"]:
                n_users_found += 1

print(f"# users: {n_users}")
print(f"# users found: {n_users_found}")
print(f"% users found: {100.0 * n_users_found / n_users:.2f}%")

# Example: explore tool calls to Git
Enjoy typing support on `entry`!

In [10]:
GIT_RE = re.compile(r"(^|[;&|\s])git(\s|$)")

def parse_main_git_call(command: str) -> str | None:
    for segment in command.split("&&"):
        segment = segment.strip()
        if segment == "git":
            return "git"
        if segment.startswith("git "):
            parts = segment.split()
            return f"{parts[0]} {parts[1]}" if len(parts) > 1 else "git"
    return None

n_tool_calls = 0
n_valid_arguments = 0
n_git_calls = 0
git_call_counts: dict[str, int] = {}

for path in table_path["agent_session_logs"]:
    parquet_file = pq.ParquetFile(path)
    for batch in tqdm(
        parquet_file.iter_batches(batch_size=batch_size),
        desc=path.name,
        total=(parquet_file.metadata.num_rows + batch_size - 1) // batch_size,
        position=0,
        leave=True,
    ):
        for row in batch.to_pylist():
            entry = cast_record(AgentSessionLogEntry, row)
            if not entry["parsed"] or entry["data"] is None:
                continue

            for choice in entry["data"]["choices"] or []:
                for tool_call in choice["delta"]["tool_calls"] or []:
                    if tool_call["function_name"] != "bash":
                        continue

                    n_tool_calls += 1

                    args_raw = tool_call["function_arguments"]
                    if not args_raw:
                        continue
                    try:
                        args = json.loads(args_raw)
                    except json.JSONDecodeError:
                        continue
                    if not isinstance(args, dict):
                        continue

                    n_valid_arguments += 1
                    command = args.get("command")
                    if not command or not GIT_RE.search(command):
                        continue

                    n_git_calls += 1
                    git_call = parse_main_git_call(command)
                    if git_call is None:
                        continue
                    git_call_counts[git_call] = git_call_counts.get(git_call, 0) + 1

print(f"# bash tool calls: {n_tool_calls}")
print(f"# valid arguments: {n_valid_arguments}")
print(f"# git calls: {n_git_calls}")
print(f"# different git calls: {len(git_call_counts)}")
dict(sorted(git_call_counts.items(), key=lambda item: item[1], reverse=True))

part_00002_of_00276.parquet:  50%|█████     | 12/24 [00:03<00:03,  3.08it/s]


KeyboardInterrupt: 